# SDialog dependencies

In [ ]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")

    # Installing sdialog
    !git clone https://github.com/qanastek/sdialog.git
    %cd sdialog
    %pip install -e .
    !pip install -q kokoro>=0.9.4
    !apt-get -qq -y install espeak-ng > /dev/null 2>&1
    %cd ..
else:
    print("Running in Jupyter Notebook")
    # Little hack to avoid the "OSError: Background processes not supported." error in Jupyter notebooks"
    get_ipython().system = os.system

## Local installation

Create a `.venv` using the root `requirement.txt` file and Python `3.11.14`

In [ ]:
from sdialog import Dialog
from IPython.display import display

# Load an existing dialogue

In order to run the next steps in a fast manner, we will start from an existing dialog generated using previous tutorials:

In [ ]:
path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog) and not os.path.exists("./demo_dialog_doctor_patient.json"):
    !wget https://raw.githubusercontent.com/qanastek/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

original_dialog = Dialog.from_file(path_dialog)
original_dialog.print()

In [ ]:
import sdialog

In [ ]:
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "XXX="

In [ ]:
sdialog.config.llm(
    "amazon:anthropic.claude-3-5-sonnet-20240620-v1:0", 
    temperature=0.7,
    max_tokens=512,
    region_name="us-east-1"
)

# Tutorial 1: Annotating Data

The key objective of this tutorial is to apply different microphone impulse responses to the audio obtains after the accoustics simulation of the room, allowing you to hear how the dialogue would sound as if recorded on various real-world devices.

In [ ]:
print(original_dialog.get_annotations())

In [ ]:
from sdialog.audio.dialog import AudioDialog

In [ ]:
test_dialog = AudioDialog.from_dialog(original_dialog)
print(test_dialog.get_annotations())

Convert the original dialog into a audio enhanced dialog

In [ ]:
from sdialog.audio.normalizers import LowercaseNormalizer, WhisperNormalizer, ReplaceCommaWithDotNormalizer, StageNormalizer
normalizers = [ReplaceCommaWithDotNormalizer(), LowercaseNormalizer(), WhisperNormalizer()]

In [ ]:
from sdialog.audio.tts import KokoroTTS
tts_engine = KokoroTTS(text_normalizers=normalizers)

In [ ]:
dialog: AudioDialog = original_dialog.to_audio(tts_engine=tts_engine, overlap_pauses=True)

In [ ]:
print(dialog.get_annotations())

In [ ]:
from sdialog.tasks.nlp import QuestionAnsweringTask

In [ ]:
annotated_qa = QuestionAnsweringTask().run(dialog)
print(annotated_qa.get_annotations())

In [ ]:
from sdialog.tasks.audio import SpokenQuestionAnsweringTask

In [ ]:
annotated_sqa = SpokenQuestionAnsweringTask().run(annotated_qa)
print(annotated_sqa.get_annotations())

If the QuestionAnswering stage wasn't run before:

In [ ]:
new_dialog: AudioDialog = original_dialog.to_audio(
    perform_room_acoustics=True,
    tts_engine=tts_engine,
    dialog_dir_name="no_question_answering",
    overlap_pauses=True
)

new_dialog_asr = new_dialog.copy()

Automatic Speech Recognition:

In [ ]:
from sdialog.tasks.audio import AutomaticSpeechRecognitionTask

In [ ]:
asr_annotated_dialog = AutomaticSpeechRecognitionTask().run(
    new_dialog_asr,
    args={
        "output_dir": "./outputs_annotations_asr",
        "save_path": "./outputs_annotations_asr/automatic_speech_recognition.csv"
    }
)
print(asr_annotated_dialog.get_annotations())

Spoken Question Answering (SQA):

In [ ]:
annotated_sqa_missing = SpokenQuestionAnsweringTask().run(new_dialog)
print(annotated_sqa_missing.get_annotations())

You can also apply multiple tasks at the same time:

In [ ]:
new_dialog2: AudioDialog = original_dialog.to_audio(dialog_dir_name="no_question_answering_bis", overlap_pauses=True)
# Duplicate the dialog to avoid modifying the original one.
new_dialog3 = new_dialog2.copy()
new_dialog3.id = "ojighifdhguyhfduighidfguidgjodfi"
new_dialog4 = new_dialog2.copy()
new_dialog4.id = "izyiuernvfuhsizoogspdgfsdbuatvev"

In [ ]:
from sdialog.tasks import dialog2tasks
from sdialog.tasks.nlp import SummarizationTask

In [ ]:
new_dialog2 = dialog2tasks(
    new_dialog2,
    [
        (QuestionAnsweringTask(), {"save_path": "./outputs_to_audio/question_answering.csv"}),
        (SummarizationTask(), {"save_path": "./outputs_to_audio/summary.csv"}),
        (SpokenQuestionAnsweringTask(), {"save_path": "./outputs_to_audio/spoken_question_answering.csv"})
    ]
)
print(new_dialog2.get_annotations())

We can also apply multiple tasks to **multiple dialogs** at the same time:

In [ ]:
from sdialog.tasks import dialogs2tasks

In [ ]:
new_dialogs = dialogs2tasks(
    [new_dialog3, new_dialog4, new_dialog2],
    [
        (QuestionAnsweringTask(), {"output_dir": "./outputs_annotations"}),
        (SummarizationTask(), {"output_dir": "./outputs_annotations"}),
        (SpokenQuestionAnsweringTask(), {"output_dir": "./outputs_annotations"})
    ]
)
print(new_dialogs[0].get_annotations())
print(new_dialogs[1].get_annotations())
print(new_dialogs[2].get_annotations())

### Named Entity Recognition

In [ ]:
medical_ner_tags = [
    # Core Clinical Entities
    "SYMPTOM",          # A subjective experience of a potential health issue (e.g., "headaches", "pain", "tired").
    "SIGN",             # An objective, observable medical fact (e.g., "tense muscles", "normal blood pressure").
    "DISEASE",          # A specific-named medical condition.
    "ANATOMY",          # A body part or location (e.g., "head", "eyes", "neck").
    "PROCEDURE",        # A medical action or test performed (e.g., "check your blood pressure").

    # Treatment and Management
    "TREATMENT",        # A recommended action to improve a condition (e.g., "stretching", "better sleep habits").
    "MEDICATION",       # A specific drug or substance prescribed.

    # People and Roles
    "PERSON",           # The name of a person (e.g., "Marie").
    "ROLE",             # The professional role of a person (e.g., "doctor").

    # Temporal Information
    "DURATION",         # The length of time a symptom has been present (e.g., "two weeks").
    "DATE",             # A specific date or time.

    # Modifiers and Other Context
    "LIFESTYLE_FACTOR"  # Factors that can influence health (e.g., "stress", "sleep").
]

In [ ]:
medical_intents = [
    "inform_symptom",
    "provide_clarification",
    "answer_question_lifestyle",
    "ask_for_diagnosis",
    "affirm",
    "thank_you",
    "ask_for_clarification",
    "propose_procedure",
    "provide_diagnosis",
    "recommend_treatment"
]

medical_slots = [
    "SYMPTOM",
    "DURATION",
    "ANATOMY",
    "LIFESTYLE_FACTOR",
    "SIGN",
    "TREATMENT",
    "PERSON"
]

In [ ]:
from sdialog.tasks.nlp import NamedEntityRecognitionTask
from sdialog.tasks.audio import SpokenLanguageUnderstandingTask, DiarizationTask, DiarizationEnhancedTask, SpeakerIdentificationTask

In [ ]:
new_dialogs = dialogs2tasks(
    [new_dialog],
    [
        (NamedEntityRecognitionTask(tags=medical_ner_tags), {"output_dir": "./outputs_annotations"}),
        (SpokenLanguageUnderstandingTask(intents=medical_intents, slots=medical_slots), {"output_dir": "./outputs_slu_annotations"}),
        (DiarizationTask(), {"output_dir": "./outputs_diarization_annotations"}),
        (DiarizationEnhancedTask(), {"output_dir": "./outputs_diarization_enhanced_annotations"}),
        (SpeakerIdentificationTask(), {"output_dir": "./outputs_speaker_identification_annotations"}),
    ]
)